In [2]:
import json



class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(NpEncoder, self).default(obj)

class dotdict(dict):
    """dot.notation access to dictionary attributes"""
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__

def load_json(file):
    with open(file) as f:
        obj = json.load(f)
    return obj


def save_json(obj, path):
    with open(path, 'w') as f:
        json.dump(obj, f, cls=NpEncoder, indent=4)



sizes=["96", "32", "48", "64", "16"]
k_values=["0.05", "0.10", "0.15", "0.20", "0.25"]
wavelets=["haar", "db1", "db2", "coif1", "coif2"]
dataset = "PMU"
best_validation_loss = 1e9
best_auc = 0.0

best_auc_config = {"dataset": dataset, "window": 0, "wavelet": 0, "k": 0}
best_val_loss_config = {"dataset": dataset, "window": 0, "wavelet": 0, "k": 0}
for size in sizes:
    for k_value in k_values:
        for wavelet in wavelets:
            auc_sum = 0
            val_loss_sum = 0
            for seed in [6, 7, 8]:
                if dataset == "PMU":
                    results = load_json(f"../log/name=window{size}_wavelet{wavelet}_k{k_value}_{dataset}_seed_{seed}/results")
                else:
                    results = load_json(f"../log/window{size}_wavelet{wavelet}_k{k_value}_{dataset}_seed_{seed}/results")
                
                validation_loss = results["best val_loss:"]
                auc = results["best_auc"]

                auc_sum += auc
                val_loss_sum += validation_loss

                
            avg_auc = auc_sum / 3
            avg_val_loss = val_loss_sum / 3

            if avg_auc > best_auc:
                best_auc = avg_auc
                best_auc_config["wavelet"] = wavelet
                best_auc_config["k"] = k_value
                best_auc_config["window"] = size
                print(avg_val_loss)

            if avg_val_loss < best_validation_loss:
                best_validation_loss = avg_val_loss
                best_val_loss_config["wavelet"] = wavelet
                best_val_loss_config["k"] = k_value
                best_val_loss_config["window"] = size


print(best_val_loss_config)
print(best_auc_config)
print(best_auc)

                


-341.1319016166355
-327.3146700651749
-367.34792051453525
-368.45697906051856
{'dataset': 'PMU', 'window': '16', 'wavelet': 'coif1', 'k': '0.05'}
{'dataset': 'PMU', 'window': '16', 'wavelet': 'db2', 'k': '0.10'}
0.9038752840016738
